<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0;">Lab 02: Create Your First Hosted Agent with MAF</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">Build a custom travel concierge by extending BaseAgent and serving it with the Foundry hosting adapter</p>
</div>

**What We Learn:** How to build a custom hosted agent by extending `BaseAgent`, implementing `run()` and `run_stream()`, and serving it locally with the Foundry hosting adapter — establishing the Contoso Travel concierge as a custom Python agent.

---

## 🧳 The Contoso Travel Story

In the prompted agents track, Contoso Travel's concierge agent was defined entirely via SDK instructions — powerful but limited. Now the engineering team wants to embed custom business logic directly in the agent: loyalty tier greetings, personalized destination recommendations based on travel history, and integration with internal booking APIs.

- **The Problem:** The prompted agent can't run custom Python code inside its process. Every enhancement requires working within the SDK's instruction-based paradigm.
- **This Lab Solves:** Building the Contoso Travel concierge as a custom `BaseAgent` that runs your own Python code, giving full control over how the agent processes requests and generates responses.

## 1. Setup

Connect to the Microsoft Foundry project and configure the OpenAI client for model calls.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #0078d4; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>Note:</b> We reuse the same <code>.env</code> file and Foundry project from Lab 01. The <code>AIProjectClient</code> provides the OpenAI-compatible client we need for chat completions.
</div>

---

In [ ]:
# Connect to Microsoft Foundry and get OpenAI client
import os
import json
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Connected to Foundry project")
print(f"   Model: {model_name}")

## 2. Define the Concierge Agent

The Microsoft Agent Framework (MAF) provides <code>BaseAgent</code> as the foundation for custom agents. You implement a single method — <code>run()</code> — which receives a list of <code>ChatMessage</code> objects and returns an <code>AgentRunResponse</code>.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #533483; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>Key concept:</b> Unlike prompted agents (defined via SDK instructions), a <code>BaseAgent</code> subclass gives you full control. You can run arbitrary Python — call APIs, apply business rules, transform data — before and after the model call.
</div>

### Mental Model: BaseAgent → run() → AgentRunResponse

```
┌─────────────────────────────┐
│  Client sends messages      │
│  (list of ChatMessage)      │
└──────────────┬──────────────┘
               │
               ▼
┌─────────────────────────────┐
│  YourAgent(BaseAgent)       │
│    def run(messages):       │
│      1. Apply business logic│
│      2. Call the model      │
│      3. Process response    │
│      return AgentRunResponse│
└──────────────┬──────────────┘
               │
               ▼
┌─────────────────────────────┐
│  AgentRunResponse           │
│  (list of ChatMessage with  │
│   role=ASSISTANT)           │
└─────────────────────────────┘
```

---

In [ ]:
# Define the Contoso Travel concierge as a BaseAgent subclass
from agent_framework import BaseAgent, AgentRunResponse, ChatMessage, Role, TextContent

# System instructions for the Contoso Travel concierge
CONCIERGE_INSTRUCTIONS = """You are the Contoso Travel concierge agent. You help customers plan trips by:
- Understanding their travel needs (destination, dates, budget, preferences)
- Providing personalized recommendations for flights, hotels, and car rentals
- Suggesting complete trip itineraries
- Being friendly, professional, and knowledgeable about all Contoso Travel destinations

Contoso Travel operates routes between: Seattle↔Paris, NYC↔London, SF↔Tokyo, Chicago↔Rome, Denver↔Cancún.
Always mention that you're part of the Contoso Travel team."""


class ContosoTravelConcierge(BaseAgent):
    """Custom concierge agent for Contoso Travel."""

    async def run(
        self, messages: list[ChatMessage], *, thread=None, **kwargs
    ) -> AgentRunResponse:
        # Prepend system instructions to the conversation
        system_msg = ChatMessage(
            role=Role.SYSTEM, content=[TextContent(text=CONCIERGE_INSTRUCTIONS)]
        )
        full_messages = [system_msg] + list(messages)

        # Call the model via the OpenAI client
        response = openai_client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": m.role.value, "content": m.content[0].text}
                for m in full_messages
            ],
        )

        # Wrap the response in AgentRunResponse format
        reply_text = response.choices[0].message.content
        return AgentRunResponse(
            messages=[
                ChatMessage(
                    role=Role.ASSISTANT, content=[TextContent(text=reply_text)]
                )
            ]
        )


concierge = ContosoTravelConcierge()
print("✅ ContosoTravelConcierge agent defined")

## 3. Test the Agent Locally (Direct Call)

Before serving the agent over HTTP, we can test it directly by calling <code>run()</code>. This is the fastest way to iterate on agent logic.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #e94560; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>Why test locally first?</b> Calling <code>run()</code> directly skips the HTTP layer, so you can debug agent logic without dealing with networking or deployment. The same code runs identically when served via the hosting adapter.
</div>

---

In [ ]:
# Test the agent locally by calling run() directly
import asyncio

# Create a test message
test_message = ChatMessage(
    role=Role.USER,
    content=[
        TextContent(
            text="I want to plan a trip from Seattle to Paris for two weeks in July. What do you recommend?"
        )
    ],
)

# Call the agent directly
response = await concierge.run([test_message])

# Display the response
print("🧳 Concierge Response:")
print("-" * 50)
print(response.messages[0].content[0].text)

## 4. Add Streaming Support

For a responsive user experience, agents should stream responses word by word. MAF supports this via <code>run_stream()</code>, an async generator that yields <code>AgentRunResponseChunk</code> objects as the model produces tokens.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #0078d4; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>Streaming pattern:</b> Implement both <code>run()</code> (for batch use) and <code>run_stream()</code> (for real-time UX). The hosting adapter automatically routes requests to the correct method based on the client's preference.
</div>

---

In [ ]:
# Concierge agent with both batch and streaming support
from agent_framework import AgentRunResponseChunk, TextContentDelta


class ContosoTravelConciergeStreaming(BaseAgent):
    """Concierge agent with streaming support."""

    async def run(
        self, messages: list[ChatMessage], *, thread=None, **kwargs
    ) -> AgentRunResponse:
        system_msg = ChatMessage(
            role=Role.SYSTEM, content=[TextContent(text=CONCIERGE_INSTRUCTIONS)]
        )
        full_messages = [system_msg] + list(messages)

        response = openai_client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": m.role.value, "content": m.content[0].text}
                for m in full_messages
            ],
        )

        reply_text = response.choices[0].message.content
        return AgentRunResponse(
            messages=[
                ChatMessage(
                    role=Role.ASSISTANT, content=[TextContent(text=reply_text)]
                )
            ]
        )

    async def run_stream(self, messages: list[ChatMessage], *, thread=None, **kwargs):
        """Stream responses word by word for real-time UX."""
        system_msg = ChatMessage(
            role=Role.SYSTEM, content=[TextContent(text=CONCIERGE_INSTRUCTIONS)]
        )
        full_messages = [system_msg] + list(messages)

        # Use streaming API
        stream = openai_client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": m.role.value, "content": m.content[0].text}
                for m in full_messages
            ],
            stream=True,
        )

        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                yield AgentRunResponseChunk(
                    delta=TextContentDelta(text=chunk.choices[0].delta.content)
                )


streaming_concierge = ContosoTravelConciergeStreaming()
print("✅ Streaming concierge agent defined")

## 5. Test Streaming

Verify the streaming agent by consuming the async generator. Each chunk arrives as the model produces tokens, giving real-time feedback.

---

In [ ]:
# Test streaming output
test_message = ChatMessage(
    role=Role.USER,
    content=[TextContent(text="What are the best things to do in Tokyo?")],
)

print("🧳 Streaming Response:")
print("-" * 50)
async for chunk in streaming_concierge.run_stream([test_message]):
    print(chunk.delta.text, end="", flush=True)
print("\n" + "-" * 50)
print("✅ Stream complete")

## 6. Serve with Foundry Hosting Adapter

The <code>from_agent_framework()</code> adapter is the bridge between your custom <code>BaseAgent</code> and Foundry's Responses API. It wraps your agent in an HTTP server so the Foundry Agent Service can route client requests to it.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #533483; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>How it works:</b> <code>from_agent_framework(agent).run()</code> starts an HTTP server on port 8088. The Foundry Agent Service connects to this endpoint, translating Responses API calls into <code>run()</code> / <code>run_stream()</code> invocations on your agent. No code changes needed — the same agent class works locally and in production.
</div>

---

In [ ]:
# In a production deployment, you'd serve the agent like this:
# from azure.ai.agentserver.agentframework import from_agent_framework
# from_agent_framework(streaming_concierge).run()  # Starts HTTP server on port 8088
#
# For this notebook, we tested the agent directly via run() and run_stream().
# The hosting adapter wraps the exact same agent — no code changes needed.

print("📋 Production deployment pattern:")
print("   from_agent_framework(ContosoTravelConcierge()).run()")
print("   → Starts HTTP server on port 8088")
print("   → Foundry Agent Service connects to this endpoint")
print("   → Clients interact via the standard Responses API")

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #e94560; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>Production deployment:</b> In a real scenario, <code>azd agent deploy</code> handles containerization, pushes your agent image to Azure Container Registry, and configures the Foundry Agent Service to route traffic to it. The agent code you wrote above is the <em>only</em> code you need — everything else is infrastructure.
</div>

## 7. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2ecc71; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>✅ What we accomplished:</b>
  <ul>
    <li>Defined a custom Contoso Travel concierge by extending <code>BaseAgent</code></li>
    <li>Implemented <code>run()</code> for batch responses and <code>run_stream()</code> for real-time streaming</li>
    <li>Tested the agent locally with direct calls — no HTTP server needed</li>
    <li>Learned the <code>from_agent_framework()</code> pattern for production deployment</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #e67e22; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>🔜 Next — Lab 03:</b> Add tools to the concierge agent so it can search flights, check hotel availability, and book trips using MAF's tool integration.
</div>

---

## Cleanup

In [ ]:
# Close client connections and release resources
openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed.")